# Experiment No. 5 — Installation and Demonstration of Data Loading into Sink Using SQL/PostgreSQL

## Aim

To install and configure a PostgreSQL database environment and load a transformed CSV file into a PostgreSQL database, demonstrating the Loading (L) stage of an ETL pipeline.


## Theory

PostgreSQL is an open-source relational database management system used to store and manage structured data. In this experiment, Pandas was used to read the transformed CSV file, while PostgreSQL was used as the final storage destination.

The `psycopg2` library was used to establish communication between Python and PostgreSQL. A database named `weatherdb` and a table named `weather_data` were created. The CSV records were then inserted into the PostgreSQL table using SQL INSERT statements.

The data was finally verified using SQL SELECT queries. This demonstrates the Loading (L) stage of an ETL pipeline, where transformed data is stored in a database for future use and analysis.

In [ ]:
import pandas as pd

print("Pandas imported successfully.")

Pandas imported successfully.


In [ ]:
import pandas as pd
import numpy as np

# Create 168 hourly weather records (7 days × 24 hours)
hours = pd.date_range("2026-08-01 00:00:00", periods=168, freq="H")

# Generate realistic Pune temperatures
np.random.seed(42)
temperature = np.round(np.random.normal(loc=27, scale=3, size=168), 1)

# Create DataFrame
df = pd.DataFrame({
    "Time": hours.strftime("%Y-%m-%d %H:%M:%S"),
    "Temperature": temperature
})

# Save CSV
df.to_csv("/content/pune_weather_analysis.csv", index=False)

print("CSV created successfully!")
display(df.head())

CSV created successfully!


/tmp/ipykernel_1313/3524160383.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range("2026-08-01 00:00:00", periods=168, freq="H")


,Time,Temperature
0,2026-08-01 00:00:00,28.5
1,2026-08-01 01:00:00,26.6
2,2026-08-01 02:00:00,28.9
3,2026-08-01 03:00:00,31.6
4,2026-08-01 04:00:00,26.3


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'pune_weather_analysis.csv', 'sample_data']


In [ ]:
import pandas as pd

df = pd.read_csv("/content/pune_weather_analysis.csv")

print("Shape:", df.shape)
display(df.head())

Shape: (168, 2)


,Time,Temperature
0,2026-08-01 00:00:00,28.5
1,2026-08-01 01:00:00,26.6
2,2026-08-01 02:00:00,28.9
3,2026-08-01 03:00:00,31.6
4,2026-08-01 04:00:00,26.3


In [ ]:
# Check whether PostgreSQL is already installed

!psql --version

/bin/bash: line 1: psql: command not found


In [ ]:
# Install PostgreSQL
!apt-get update -qq
!apt-get install -y postgresql postgresql-contrib

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libllvm14 libpq-dev libpq5
  libtypes-serialiser-perl logrotate netbase postgresql-14
  postgresql-client-14 postgresql-client-common postgresql-common ssl-cert
  sysstat
Suggested packages:
  postgresql-doc-14 bsd-mailx | mailx postgresql-doc isag
The following NEW packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libllvm14
  libtypes-serialiser-perl logrotate netbase postgresql postgresql-14
  postgresql-client-14 postgresql-client-common postgresql-common
  postgresql-contrib ssl-cert sysstat
The following packages will be upgraded:
  libpq-dev libpq5
2 upgraded, 15 new

In [ ]:
# Start PostgreSQL service
!service postgresql start

# Check PostgreSQL version
!psql --version

 * Starting PostgreSQL 14 database server
   ...done.
psql (PostgreSQL) 14.24 (Ubuntu 14.24-0ubuntu0.22.04.1)


In [ ]:
# Create the PostgreSQL database

!sudo -u postgres psql -c "CREATE DATABASE weatherdb;"

CREATE DATABASE


In [ ]:
# Install PostgreSQL Python connector
!pip install -q psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 24.0 MB/s eta 0:00:00


In [ ]:
import psycopg2

print("psycopg2 imported successfully!")

psycopg2 imported successfully!


In [ ]:
# Connect Python to PostgreSQL

import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="weatherdb",
    user="postgres",
    password="postgres"
)

print("Connected to PostgreSQL successfully!")

Connected to PostgreSQL successfully!


In [ ]:
# Set password for PostgreSQL postgres user

!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"

print("PostgreSQL password configured.")

ALTER ROLE
PostgreSQL password configured.


In [ ]:
# Create weather_data table in PostgreSQL

cursor = conn.cursor()

create_table_query = """
CREATE TABLE IF NOT EXISTS weather_data (
    id SERIAL PRIMARY KEY,
    time TIMESTAMP,
    temperature FLOAT
);
"""

cursor.execute(create_table_query)
conn.commit()

print("weather_data table created successfully!")

weather_data table created successfully!


In [ ]:
# Check the table structure

cursor.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'weather_data';
""")

table_info = cursor.fetchall()

for column in table_info:
    print(column)

('id', 'integer')
('time', 'timestamp without time zone')
('temperature', 'double precision')


In [ ]:
# Load CSV data into PostgreSQL

# Make sure time is in datetime format
df["Time"] = pd.to_datetime(df["Time"])

for _, row in df.iterrows():
    cursor.execute(
        """
        INSERT INTO weather_data (time, temperature)
        VALUES (%s, %s)
        """,
        (row["Time"], row["Temperature"])
    )

conn.commit()

print("CSV data loaded into PostgreSQL successfully!")
print("Records loaded:", len(df))

CSV data loaded into PostgreSQL successfully!
Records loaded: 168


In [ ]:
# Verify loaded data

cursor.execute("SELECT COUNT(*) FROM weather_data;")
count = cursor.fetchone()[0]

print("Total records in PostgreSQL:", count)

print("\nFirst 5 records:")
cursor.execute("""
    SELECT * FROM weather_data
    ORDER BY id
    LIMIT 5;
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

Total records in PostgreSQL: 168

First 5 records:
(1, datetime.datetime(2026, 8, 1, 0, 0), 28.5)
(2, datetime.datetime(2026, 8, 1, 1, 0), 26.6)
(3, datetime.datetime(2026, 8, 1, 2, 0), 28.9)
(4, datetime.datetime(2026, 8, 1, 3, 0), 31.6)
(5, datetime.datetime(2026, 8, 1, 4, 0), 26.3)


In [ ]:
# Run SQL query for basic weather analysis

query = """
SELECT
    COUNT(*) AS total_records,
    ROUND(AVG(temperature)::numeric, 2) AS average_temperature,
    MIN(temperature) AS minimum_temperature,
    MAX(temperature) AS maximum_temperature
FROM weather_data;
"""

cursor.execute(query)

result = cursor.fetchone()

print("Weather Data Summary")
print("--------------------")
print("Total Records:", result[0])
print("Average Temperature:", result[1], "°C")
print("Minimum Temperature:", result[2], "°C")
print("Maximum Temperature:", result[3], "°C")

Weather Data Summary
--------------------
Total Records: 168
Average Temperature: 26.88 °C
Minimum Temperature: 19.1 °C
Maximum Temperature: 34.4 °C


## Conclusion

The PostgreSQL database environment was successfully installed and configured in Google Colab. The transformed weather CSV file was read using Pandas and loaded into the PostgreSQL `weatherdb` database. A `weather_data` table was created and all 168 records were successfully inserted and verified using SQL queries. Thus, the Loading stage of the ETL pipeline was successfully demonstrated.